# Финальный проект: рекомендация банковских продуктов

**Датасет:** Santander Product Recommendation (`train_ver2.csv`).

**Ограничения:** из-за RAM EDA и обучение идут на срезе **2016 года** и **доли клиентов** (не на полном датасете ~2.3 GB). Доли задаются в **следующей ячейке** (`CLIENT_SAMPLE_FRAC`, `MODEL_CLIENT_FRAC`); при достаточной памяти увеличьте до 0.5–1.0.

**Структура ноутбука:** параметры → EDA → выводы → моделирование → MLflow. Ячейки выполняйте **сверху вниз**.




In [3]:
# Общие параметры и MLflow (выполните эту ячейку первой)
import os
from dotenv import load_dotenv
import mlflow

load_dotenv()

CLIENT_SAMPLE_FRAC = 0.3   # доля клиентов за 2016; при RAM > 16GB: 0.5–1.0
MODEL_CLIENT_FRAC = CLIENT_SAMPLE_FRAC  # EDA и модель на одном срезе
RANDOM_SEED = 42

# --- MLflow + Yandex Object Storage (как в mlflow_server/run_mlflow_server.sh) ---
MLFLOW_TRACKING_URI = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')
MLFLOW_EXPERIMENT = os.getenv('MLFLOW_EXPERIMENT_NAME', 'bank_product_recommendation')

os.environ['MLFLOW_S3_ENDPOINT_URL'] = os.getenv(
    'MLFLOW_S3_ENDPOINT_URL', 'https://storage.yandexcloud.net'
)
for _key in ('AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY'):
    _val = os.getenv(_key)
    if _val:
        os.environ[_key] = _val.strip().strip('"').strip("'")

_bucket = os.getenv('S3_BUCKET_NAME') or os.getenv('AWS_BUCKET_NAME')
if _bucket:
    os.environ['AWS_BUCKET_NAME'] = _bucket.strip()

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

print(f'CLIENT_SAMPLE_FRAC = {CLIENT_SAMPLE_FRAC}')
print(f'MODEL_CLIENT_FRAC  = {MODEL_CLIENT_FRAC}')
print(f'RANDOM_SEED          = {RANDOM_SEED}')
print(f'MLFLOW_TRACKING_URI  = {MLFLOW_TRACKING_URI}')
print(f'MLFLOW_S3_ENDPOINT   = {os.environ["MLFLOW_S3_ENDPOINT_URL"]}')

if not os.getenv('AWS_ACCESS_KEY_ID') or not os.getenv('AWS_SECRET_ACCESS_KEY'):
    print('⚠️  Заполните AWS_ACCESS_KEY_ID и AWS_SECRET_ACCESS_KEY в .env (ключи Yandex Cloud)')
else:
    print(f'AWS_ACCESS_KEY_ID    = {os.getenv("AWS_ACCESS_KEY_ID")[:8]}...')
    print(f'S3_BUCKET_NAME       = {_bucket or "не задан"}')

CLIENT_SAMPLE_FRAC = 0.3
MODEL_CLIENT_FRAC  = 0.3
RANDOM_SEED          = 42
MLFLOW_TRACKING_URI  = http://localhost:5000
MLFLOW_S3_ENDPOINT   = https://storage.yandexcloud.net
AWS_ACCESS_KEY_ID    = YCAJE3Nl...
S3_BUCKET_NAME       = s3-student-mle-20251208-9fda70ca95-freetrack


In [4]:
# -*- coding: utf-8 -*-
"""
EDA для датасета банковских продуктов Santander.
Срез данных согласован с блоком моделирования (2016 год, доля клиентов).
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
from matplotlib.backends.backend_pdf import PdfPages
import warnings
warnings.filterwarnings('ignore')

EDA_REPORT_PDF = 'eda_bank_products_report.pdf'

# CLIENT_SAMPLE_FRAC, RANDOM_SEED — из ячейки «Общие параметры» выше
NA_VALUES = [' NA', '     NA', 'NA', 'n/a', 'N/A', 'NULL', 'null', '', ' ', '?', 'unknown']

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('=' * 70)
print(f'EDA: 2016 год, {CLIENT_SAMPLE_FRAC:.0%} клиентов (как в моделировании)')
print('=' * 70)

sample_headers = pd.read_csv('data/train_ver2.csv', nrows=0)
target_cols = [
    col for col in sample_headers.columns
    if col.startswith('ind_') and col.endswith('_ult1')
]
print(f'Целевых продуктов: {len(target_cols)}')

usecols = [
    'fecha_dato', 'ncodpers', 'age', 'antiguedad', 'renta', 'sexo',
    'segmento', 'ind_empleado', 'canal_entrada',
] + target_cols

dtypes = {
    'fecha_dato': 'object', 'ncodpers': 'int32', 'age': 'float32',
    'antiguedad': 'float32', 'renta': 'float32', 'sexo': 'object',
    'segmento': 'object', 'ind_empleado': 'object', 'canal_entrada': 'object',
}

df = pd.read_csv(
    'data/train_ver2.csv',
    usecols=usecols,
    dtype=dtypes,
    na_values=NA_VALUES,
    skipinitialspace=True,
    low_memory=False,
)
df['fecha_dato'] = pd.to_datetime(df['fecha_dato'])

for col in target_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('int8')

df = df[df['fecha_dato'] >= '2016-01-01']
print(f'Записей за 2016 (до семпла клиентов): {len(df):,}')

np.random.seed(RANDOM_SEED)
unique_clients = df['ncodpers'].unique()
selected_clients = np.random.choice(
    unique_clients,
    size=int(len(unique_clients) * CLIENT_SAMPLE_FRAC),
    replace=False,
)
df = df[df['ncodpers'].isin(selected_clients)].copy()
print(f'После выборки {CLIENT_SAMPLE_FRAC:.0%} клиентов: {len(df):,} строк')
print(f'Уникальных клиентов: {df["ncodpers"].nunique():,}')
print(f'Память: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

cat_cols = ['sexo', 'segmento', 'ind_empleado', 'canal_entrada']
for col in cat_cols:
    df[col] = df[col].astype('category')

for col in ['age', 'antiguedad', 'renta']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('float32')

print('\n' + '=' * 70)
print('ИНФОРМАЦИЯ О ДАННЫХ')
print('=' * 70)
print(df[['age', 'renta', 'antiguedad'] + cat_cols].info())
print(f"\nПропуски в ключевых колонках:\n{df[['age', 'renta', 'antiguedad']].isnull().sum()}")

df['num_products'] = df[target_cols].sum(axis=1)
print(f"\nСреднее количество продуктов: {df['num_products'].mean():.2f}")
print(f"Медиана: {df['num_products'].median()}")
print(f"Максимум: {df['num_products'].max()}")

product_freq = df[target_cols].sum().sort_values(ascending=False)
top_products = product_freq.head(15)
print(f"\nТоп-3 продукта:\n{top_products.head(3)}")

# Дисбаланс: доля положительного класса по продуктам
prevalence = (df[target_cols].mean() * 100).sort_values(ascending=False)
print(f"\nДоля владения (%, топ-5):\n{prevalence.head()}")
print(f"Доля владения (%, редкие):\n{prevalence.tail(3)}")

fig1, ax1 = plt.subplots()
sns.histplot(df['num_products'], bins=range(0, int(df['num_products'].max()) + 2), discrete=True, ax=ax1)
ax1.set_title('Распределение количества продуктов на клиента')

fig2, ax2 = plt.subplots(figsize=(12, 6))
top_products.plot(kind='bar', color='skyblue', ax=ax2)
ax2.set_title('Топ-15 наиболее распространённых продуктов')
ax2.tick_params(axis='x', rotation=45)

top20 = product_freq.head(20).index
jaccard_mat = np.zeros((len(top20), len(top20)))
for i, p1 in enumerate(top20):
    for j, p2 in enumerate(top20):
        jaccard_mat[i, j] = (df[p1] & df[p2]).sum() / (df[p1] | df[p2]).sum()

fig3, ax3 = plt.subplots(figsize=(12, 10))
sns.heatmap(jaccard_mat, xticklabels=top20, yticklabels=top20, cmap='Blues', annot=False, ax=ax3)
ax3.set_title('Сходство Жаккара (топ-20 продуктов)')

monthly = df.groupby(df['fecha_dato'].dt.to_period('M')).agg({
    'num_products': 'mean',
    **{p: 'mean' for p in top_products.head(5).index},
}).reset_index()
monthly['fecha_dato'] = monthly['fecha_dato'].astype(str)

fig4, ax4 = plt.subplots(figsize=(14, 6))
for prod in top_products.head(5).index:
    ax4.plot(monthly['fecha_dato'], monthly[prod], marker='o', label=prod)
ax4.set_title('Динамика проникновения топ-5 продуктов')
ax4.legend()
plt.xticks(rotation=45)

top3 = top_products.head(3).index
for prod in top3:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].boxplot(
        [df[df[prod] == 1]['age'].dropna(), df[df[prod] == 0]['age'].dropna()],
        labels=['Есть', 'Нет'],
    )
    axes[0].set_title(f'Возраст vs {prod}')
    axes[1].boxplot(
        [df[df[prod] == 1]['renta'].dropna(), df[df[prod] == 0]['renta'].dropna()],
        labels=['Есть', 'Нет'],
    )
    axes[1].set_title(f'Доход vs {prod}')
    axes[2].boxplot(
        [df[df[prod] == 1]['antiguedad'].dropna(), df[df[prod] == 0]['antiguedad'].dropna()],
        labels=['Есть', 'Нет'],
    )
    axes[2].set_title(f'Стаж vs {prod}')
    plt.tight_layout()

for col in cat_cols:
    fig, axes = plt.subplots(1, len(top3), figsize=(15, 4))
    for i, prod in enumerate(top3):
        grouped = df.groupby(col, observed=True)[prod].mean().sort_values(ascending=False).head(8)
        axes[i].barh(grouped.index.astype(str), grouped.values, color='teal')
        axes[i].set_title(f'Доля {prod} по {col}')
    plt.tight_layout()

sample_corr = df[['age', 'renta', 'antiguedad', 'num_products'] + list(top3)].dropna().sample(
    min(30_000, len(df)), random_state=RANDOM_SEED,
)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(sample_corr.corr(), annot=True, cmap='coolwarm', center=0, fmt='.2f', ax=ax)
ax.set_title('Корреляция числовых признаков и топ-продуктов')

with PdfPages(EDA_REPORT_PDF) as pdf:
    for i in plt.get_fignums():
        fig = plt.figure(i)
        pdf.savefig(fig)
        plt.close(fig)

print('\n' + '=' * 70)
print(f'ОТЧЁТ СОХРАНЁН: {EDA_REPORT_PDF}')
print('=' * 70)

# --- MLflow: артефакты EDA (S3 env заданы в ячейке «Общие параметры») ---
product_freq.to_csv('eda_product_frequency.csv', header=['count'])
prevalence.to_csv('eda_product_prevalence_pct.csv', header=['prevalence_pct'])

with mlflow.start_run(run_name='eda'):
    mlflow.log_params({
        'stage': 'eda',
        'client_sample_frac': CLIENT_SAMPLE_FRAC,
        'random_seed': RANDOM_SEED,
        'year_filter': '2016',
        'n_products': len(target_cols),
    })
    mlflow.log_metrics({
        'eda_rows': len(df),
        'eda_unique_clients': df['ncodpers'].nunique(),
        'mean_products_per_client': float(df['num_products'].mean()),
        'median_products_per_client': float(df['num_products'].median()),
        'max_products_per_client': float(df['num_products'].max()),
        'renta_missing_pct': float(df['renta'].isna().mean() * 100),
    })
    mlflow.log_artifact(EDA_REPORT_PDF, artifact_path='eda')
    mlflow.log_artifact('eda_product_frequency.csv', artifact_path='eda')
    mlflow.log_artifact('eda_product_prevalence_pct.csv', artifact_path='eda')
    mlflow.set_tag('mlflow.note.content', 'EDA: PDF-отчёт и частоты продуктов')

print(f"MLflow: артефакты EDA залогированы (run 'eda')")
print(f"UI: {MLFLOW_TRACKING_URI}")




EDA: 2016 год, 30% клиентов (как в моделировании)
Целевых продуктов: 24
Записей за 2016 (до семпла клиентов): 4,621,976
После выборки 30% клиентов: 1,386,596 строк
Уникальных клиентов: 281,526
Память: 403.9 MB

ИНФОРМАЦИЯ О ДАННЫХ
<class 'pandas.core.frame.DataFrame'>
Index: 1386596 entries, 9025333 to 13647305
Data columns (total 7 columns):
 #   Column         Non-Null Count    Dtype   
---  ------         --------------    -----   
 0   age            1386596 non-null  float32 
 1   renta          1055331 non-null  float32 
 2   antiguedad     1386596 non-null  float32 
 3   sexo           1386586 non-null  category
 4   segmento       1374578 non-null  category
 5   ind_empleado   1386596 non-null  category
 6   canal_entrada  1375017 non-null  category
dtypes: category(4), float32(3)
memory usage: 33.1 MB
None

Пропуски в ключевых колонках:
age                0
renta         331265
antiguedad         0
dtype: int64

Среднее количество продуктов: 1.33
Медиана: 1.0
Максимум: 14

Топ

## Выводы по EDA

1. **Структура владения продуктами.** В среднем у клиента ~1–2 продукта (медиана 1), максимум до 14. Распределение сильно скошено — типичный multi-label с редкими комбинациями.

2. **Популярные продукты.** Лидер — `ind_cco_fin_ult1` (текущий счёт). Есть длинный «хвост» редких продуктов с долей владения &lt; 1%.

3. **Связь продуктов.** Матрица Жаккара показывает, что продукты часто берут вместе (кластеры в heatmap) — обоснован подход One-vs-Rest / multi-label.

4. **Временная динамика.** По месяцам 2016 года меняется проникновение топ-продуктов → нужен **временной split**, а не случайное разбиение.

5. **Числовые признаки.** Возраст, доход (`renta`) и стаж (`antiguedad`) различают клиентов с продуктом и без (boxplot'ы) → включаем в модель.

6. **Категориальные признаки.** `segmento`, `canal_entrada`, `sexo`, `ind_empleado` влияют на долю владения продуктами.

7. **Пропуски.** `renta` ~18% пропусков → заполнение медианой по `segmento` (как в EDA-рекомендации).

8. **Дисбаланс классов.** Большинство пар «клиент–месяц–продукт» отрицательные → `auto_class_weights='Balanced'` в CatBoost.

9. **Ограничения среза.** EDA и модель строятся на **2016 году** и **доле клиентов** (`CLIENT_SAMPLE_FRAC` в первой code-ячейке) из-за ограничений RAM; при больших ресурсах долю можно увеличить до 0.5–1.0.

Артефакты EDA (`eda_bank_products_report.pdf`, CSV с частотами) сохраняются в MLflow в run **`eda`**.

### Решения для моделирования

| Наблюдение EDA | Решение в модели |
|----------------|------------------|
| Новые продукты = изменение портфеля во времени | Таргет: `product_new` через `shift(-1)` |
| Временной тренд | Train / Val / Test по месяцам 2016 |
| Пропуски `renta` | Imputation по `segmento` |
| Дисбаланс | CatBoost `auto_class_weights='Balanced'` |
| Метрика соревнования | MAP@7, дополнительно Recall@7 и PR-AUC |




## Моделирование

- **Задача:** multi-label — предсказать новые продукты в следующем месяце.
- **Модель:** `OneVsRestClassifier` + `CatBoostClassifier`.
- **Логирование:** MLflow (запустите `./mlflow_server/run_mlflow_server.sh` и проверьте `.env`).




In [5]:
# Моделирование банковских продуктов

import os
import gc
import warnings
import joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn
from catboost import CatBoostClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import average_precision_score

warnings.filterwarnings('ignore')

# MODEL_CLIENT_FRAC, RANDOM_SEED — из ячейки «Общие параметры»
NA_VALUES = [' NA', '     NA', 'NA', 'n/a', 'N/A', 'NULL', 'null', '', ' ', '?', 'unknown']

# MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT — из ячейки «Общие параметры»


def build_proba_matrix(y_pred, n_samples, n_products):
    """OneVsRest + CatBoost: (n_samples, n_products) или список массивов."""
    if isinstance(y_pred, list):
        cols = []
        for p in y_pred:
            p = np.asarray(p)
            cols.append(p[:, 1] if p.ndim == 2 and p.shape[1] == 2 else p.ravel())
        return np.column_stack(cols)

    arr = np.asarray(y_pred)
    if arr.ndim == 3:
        return arr[:, :, 1]
    if arr.ndim == 2:
        if arr.shape[0] == n_samples and arr.shape[1] == n_products:
            return arr
        if arr.shape[0] == n_products and arr.shape[1] == n_samples:
            return arr.T
    raise ValueError(f'Неожиданная форма predict_proba: {arr.shape}')


def map_at_k(y_true, y_score, k=7):
    ap = []
    for i in range(y_true.shape[0]):
        true = y_true[i]
        if true.sum() == 0:
            continue
        top = np.argsort(y_score[i])[::-1][:k]
        rel = true[top]
        cum_prec = np.cumsum(rel) / (np.arange(1, len(rel) + 1))
        ap.append(np.sum(cum_prec * rel) / min(k, true.sum()))
    return float(np.mean(ap)) if ap else 0.0


def recall_at_k(y_true, y_score, k=7):
    recalls = []
    for i in range(y_true.shape[0]):
        true = y_true[i]
        if true.sum() == 0:
            continue
        top = np.argsort(y_score[i])[::-1][:k]
        recalls.append(true[top].sum() / min(k, true.sum()))
    return float(np.mean(recalls)) if recalls else 0.0


def pr_auc_macro(y_true, y_score):
    """Средний PR-AUC по продуктам с хотя бы одним положительным примером."""
    scores = []
    for j in range(y_true.shape[1]):
        if y_true[:, j].sum() > 0:
            scores.append(average_precision_score(y_true[:, j], y_score[:, j]))
    return float(np.mean(scores)) if scores else 0.0


def evaluate_split(model, X, y, n_products, prefix):
    proba = build_proba_matrix(model.predict_proba(X), len(X), n_products)
    yv = y.values
    metrics = {
        f'{prefix}_map_at_7': map_at_k(yv, proba, k=7),
        f'{prefix}_recall_at_7': recall_at_k(yv, proba, k=7),
        f'{prefix}_pr_auc': pr_auc_macro(yv, proba),
    }
    return metrics, proba.shape


print('Загрузка данных...')
dtypes = {
    'fecha_dato': 'object', 'ncodpers': 'int32', 'age': 'float32',
    'antiguedad': 'float32', 'renta': 'float32', 'sexo': 'object',
    'segmento': 'object', 'ind_empleado': 'object', 'canal_entrada': 'object',
    'ind_nuevo': 'float32', 'indrel': 'float32',
}

sample_headers = pd.read_csv('data/train_ver2.csv', nrows=0)
target_cols = [
    col for col in sample_headers.columns
    if col.startswith('ind_') and col.endswith('_ult1')
]

usecols = [
    'fecha_dato', 'ncodpers', 'age', 'antiguedad', 'renta', 'sexo',
    'segmento', 'ind_empleado', 'canal_entrada', 'ind_nuevo', 'indrel',
] + target_cols

df = pd.read_csv(
    'data/train_ver2.csv',
    usecols=usecols,
    dtype=dtypes,
    na_values=NA_VALUES,
    skipinitialspace=True,
    low_memory=False,
)
df['fecha_dato'] = pd.to_datetime(df['fecha_dato'])

for col in target_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('int8')

df = df[df['fecha_dato'] >= '2016-01-01']
print(f'Всего записей за 2016: {len(df):,}')

np.random.seed(RANDOM_SEED)
unique_clients = df['ncodpers'].unique()
selected_clients = np.random.choice(
    unique_clients,
    size=int(len(unique_clients) * MODEL_CLIENT_FRAC),
    replace=False,
)
df = df[df['ncodpers'].isin(selected_clients)]
print(f'После выборки {MODEL_CLIENT_FRAC:.0%} клиентов: {len(df):,} записей')

df = df.sort_values(['ncodpers', 'fecha_dato']).reset_index(drop=True)
shifted = df.groupby('ncodpers')[target_cols].shift(-1)
for col in target_cols:
    df[f'{col}_new'] = ((shifted[col] == 1) & (df[col] == 0)).astype('int8')

new_target_cols = [f'{col}_new' for col in target_cols]
last_month = df.groupby('ncodpers')['fecha_dato'].transform('max')
df = df[df['fecha_dato'] < last_month].copy()
print(f'Строк после удаления последнего месяца клиента: {len(df):,}')

feature_cols = [
    'age', 'antiguedad', 'renta', 'sexo', 'segmento', 'ind_empleado', 'canal_entrada',
]
df['renta'] = df.groupby('segmento')['renta'].transform(lambda x: x.fillna(x.median()))
df['renta'] = df['renta'].fillna(df['renta'].median())
df['age'] = df['age'].fillna(df['age'].median())
df['antiguedad'] = df['antiguedad'].fillna(0)

cat_features = ['sexo', 'segmento', 'ind_empleado', 'canal_entrada']
for col in cat_features:
    df[col] = df[col].astype(str).fillna('unknown')

train_mask = df['fecha_dato'].isin(['2016-01-28', '2016-02-28'])
val_mask = df['fecha_dato'] == '2016-03-28'
test_mask = df['fecha_dato'] == '2016-04-28'

X_train = df.loc[train_mask, feature_cols]
y_train = df.loc[train_mask, new_target_cols]
X_val = df.loc[val_mask, feature_cols]
y_val = df.loc[val_mask, new_target_cols]
X_test = df.loc[test_mask, feature_cols]
y_test = df.loc[test_mask, new_target_cols]
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

del df
gc.collect()

catboost_params = dict(
    iterations=50,
    learning_rate=0.1,
    depth=4,
    auto_class_weights='Balanced',
    random_seed=RANDOM_SEED,
    verbose=10,
    cat_features=cat_features,
    loss_function='Logloss',
)

base_model = CatBoostClassifier(**catboost_params)
model = OneVsRestClassifier(base_model)

with mlflow.start_run(run_name='catboost_ovr_baseline'):
    mlflow.log_params({
        'model': 'OneVsRestClassifier(CatBoostClassifier)',
        'client_sample_frac': MODEL_CLIENT_FRAC,
        'random_seed': RANDOM_SEED,
        'train_months': '2016-01,2016-02',
        'val_month': '2016-03',
        'test_month': '2016-04',
        'features': ','.join(feature_cols),
        **{f'cb_{k}': v for k, v in catboost_params.items() if k != 'verbose'},
    })
    mlflow.log_param('n_products', len(new_target_cols))

    print('Обучение начато...')
    model.fit(X_train, y_train)
    print('Обучение завершено.')

    n_products = len(new_target_cols)
    all_metrics = {}

    for split_name, X_split, y_split, prefix in [
        ('Validation', X_val, y_val, 'val'),
        ('Test', X_test, y_test, 'test'),
    ]:
        m, shape = evaluate_split(model, X_split, y_split, n_products, prefix)
        all_metrics.update(m)
        print(
            f"{split_name}: MAP@7={m[f'{prefix}_map_at_7']:.4f}, "
            f"Recall@7={m[f'{prefix}_recall_at_7']:.4f}, "
            f"PR-AUC={m[f'{prefix}_pr_auc']:.4f}"
        )

    mlflow.log_metrics(all_metrics)

    model_path = 'model.bin'
    joblib.dump(model, model_path)
    mlflow.log_artifact(model_path, artifact_path='model')
    mlflow.sklearn.log_model(model, artifact_path='sklearn_model')

    print('Модель сохранена: model.bin')
    print(f'MLflow UI: {MLFLOW_TRACKING_URI}')

trained_estimators = [
    est for est in model.estimators_
    if hasattr(est, 'feature_importances_')
]
if trained_estimators:
    avg_importance = np.mean(
        [est.feature_importances_ for est in trained_estimators], axis=0,
    )
    imp = pd.DataFrame({
        'feature': feature_cols,
        'importance': avg_importance,
    }).sort_values('importance', ascending=False)
    print(
        f'\nТоп-5 признаков (среднее по {len(trained_estimators)} обученным продуктам из {n_products}):'
    )
    print(imp.head())
    if len(trained_estimators) < n_products:
        print(
            f'Примечание: для {n_products - len(trained_estimators)} продуктов '
            'в train не было положительных примеров (ConstantPredictor).'
        )
else:
    print('Нет обученных CatBoost-моделей для важности признаков.')




Загрузка данных...
Всего записей за 2016: 4,621,976
После выборки 30% клиентов: 1,386,596 записей
Строк после удаления последнего месяца клиента: 1,105,070
Train: (549968, 7), Val: (277029, 7), Test: (278073, 7)
Обучение начато...
0:	learn: 0.6756217	total: 195ms	remaining: 9.54s
10:	learn: 0.5964934	total: 956ms	remaining: 3.39s
20:	learn: 0.5745665	total: 1.59s	remaining: 2.2s
30:	learn: 0.5615036	total: 2.33s	remaining: 1.43s
40:	learn: 0.5572769	total: 3.03s	remaining: 665ms
49:	learn: 0.5549488	total: 3.7s	remaining: 0us
0:	learn: 0.6648623	total: 88.5ms	remaining: 4.34s
10:	learn: 0.5100029	total: 779ms	remaining: 2.76s
20:	learn: 0.4464230	total: 1.39s	remaining: 1.92s
30:	learn: 0.3572181	total: 2.02s	remaining: 1.24s
40:	learn: 0.3386698	total: 2.59s	remaining: 569ms
49:	learn: 0.3190386	total: 3.08s	remaining: 0us
0:	learn: 0.6821408	total: 99.5ms	remaining: 4.88s
10:	learn: 0.6308041	total: 821ms	remaining: 2.91s
20:	learn: 0.6075692	total: 1.57s	remaining: 2.17s
30:	learn: 